In [2]:
import pandas as pd
import pyreadr
from datetime import datetime
import os

current_dir = os.getcwd()
path = os.path.abspath(os.path.join(current_dir, '../tweets/RData Tweets/'))
path_labelled = os.path.abspath(os.path.join(current_dir, '../tweets/Labelled_Tweets/'))
raw_tweets = pyreadr.read_r(path + '/raw_tweets.RData')
raw_tweets2 = pyreadr.read_r(path + '/updated_homestyle.RData')
train_tweeks = pd.read_csv(path_labelled + '/training_data_labelled_bak.csv')
test_tweets = pd.read_csv(path_labelled + '/test_data_labelled_bak.csv')

In [4]:
raw_df = raw_tweets2['updated_homestyle']
# Only english tweets
print(raw_df.columns)
raw_df = raw_df[['tweet_id', 'text', 'author_id', 'tw_date', 'created_at']]
raw_df['tw_date'] = pd.to_datetime(raw_df['tw_date'], errors='coerce')
raw_df['date'] = raw_df['tw_date'].apply(lambda x: str(x.date()))
raw_df['year'] = raw_df['tw_date'].dt.year.astype(str)
raw_df['year'] = raw_df['tw_date'].dt.year.astype(str)
raw_df = raw_df[raw_df['year'] != 'nan']
raw_df['year'] = raw_df['year'].astype(float).astype(int).astype(str)
raw_df['author_id'] = raw_df['author_id'].astype(int)


Index(['user_username', 'tweet_id', 'text', 'author_id', 'created_at',
       'retweet_count', 'like_count', 'quote_count', 'user_tweet_count',
       'user_list_count', 'tw_date', 'year', 'tw_month', 'tw_day',
       'icpsr_sen_id', 'state_abbrev', 'yr_engagement_score', 'is_homestyle'],
      dtype='object')


/var/folders/ll/wxkddfw94dq7r5058pg5h5r80000gn/T/ipykernel_374/1251824951.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_df['tw_date'] = pd.to_datetime(raw_df['tw_date'], errors='coerce')
/var/folders/ll/wxkddfw94dq7r5058pg5h5r80000gn/T/ipykernel_374/1251824951.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_df['date'] = raw_df['tw_date'].apply(lambda x: str(x.date()))
/var/folders/ll/wxkddfw94dq7r5058pg5h5r80000gn/T/ipykernel_374/1251824951.py:7: SettingWithCopyWarning: 
A value is trying 

In [40]:
import json
import datetime as datetime
train = train_tweeks[['tweet_id', 'text', 'tw_date', 'author_id', 'year', 'source', 'AR', 'MB']]
test = test_tweets[['tweet_id', 'text', 'tw_date', 'author_id', 'year', 'source', 'AR', 'MB']]

combined = pd.concat([train, test], ignore_index=True)

combined['date'] = combined['tw_date'].astype(str).str.split(' ').str[0]

combined['year'] = combined['year'].astype(str)

combined['idddd'] = range(len(combined))


In [34]:
combined

,tweet_id,text,tw_date,author_id,year,source,AR,MB,date,idddd
0,7.300000e+17,Met w/folks from General Aviation Mfg Assoc. a...,2016-05-11 00:00:00,1061029050,2016,randomized_2016,2,2,2016-05-11,0
1,7.080000e+17,Talked about the benefits of bicycling for ND ...,2016-03-09 00:00:00,1061029050,2016,randomized_2016,3,3,2016-03-09,1
2,7.820000e+17,At the Dick Morris Memorial Tailgate for the U...,2016-10-01 00:00:00,10615232,2016,randomized_2016,3,3,2016-10-01,2
3,8.040000e+17,SOON: I will speak on the #SenateFloor on the ...,2016-11-30 00:00:00,1071402577,2016,randomized_2016,3,3,2016-11-30,3
4,7.350000e+17,Enjoyed chatting w/ students from @UCollegeNE ...,2016-05-25 00:00:00,1071402577,2016,randomized_2016,3,3,2016-05-25,4
...,...,...,...,...,...,...,...,...,...,...
4199,1.250131e+18,"We must ensure that Colorado farmers, ranchers...",2020-04-14 18:37:27+00:00,Michael Bennet,2020,selected_2020,2,2,2020-04-14,4199
4200,1.250133e+18,.@SenSchumer @PattyMurray and I are calling on...,2020-04-14 18:43:16+00:00,Ron Wyden,2020,selected_2020,1,1,2020-04-14,4200
4201,1.250141e+18,"RT @TeamOssoff: If you're feeling down, you're...",2020-04-14 19:18:29+00:00,Jon Ossoff,2020,selected_2020,3,3,2020-04-14,4201
4202,1.250144e+18,"On this #414Day, I hope every Milwaukeean is s...",2020-04-14 19:30:00+00:00,Tammy Baldwin,2020,selected_2020,3,3,2020-04-14,4202


In [41]:
# Perform the merge based on the join_conditions
join_conditions = ['year', 'date']
merged_df = pd.merge(
    combined, 
    raw_df, 
    how='left', 
    left_on=join_conditions,
    right_on= join_conditions
)
print(len(merged_df['idddd'].unique()))
# merged_df.to_csv('~/Documents/hahahhaha.csv')

4204


In [43]:
from fuzzywuzzy import fuzz
merged_df_fuz = merged_df[~merged_df['text_y'].isna()]
no_match = merged_df[merged_df['text_y'].isna()]

print(len(merged_df_fuz['idddd'].unique()))
print(len(no_match['idddd'].unique()))
# Add a new column to store the similarity score
merged_df_fuz['similarity'] = merged_df_fuz.apply(lambda row: fuzz.ratio(row['text_x'], row['text_y']), axis=1)
print(len(merged_df_fuz['idddd'].unique()))
# For each text_x, keep only the row with the highest similarity score
best_match_df = merged_df_fuz.loc[merged_df_fuz.groupby(['idddd', 'text_x'])['similarity'].idxmax()]
print(len(best_match_df['idddd'].unique()))
bad_match = best_match_df[best_match_df['similarity'] < 80]
best_match_df = best_match_df[best_match_df['similarity'] >= 80]
print(no_match.shape)
print(best_match_df.shape)
no_match = pd.concat([no_match, bad_match], ignore_index=True)
no_match.to_csv('~/Downloads/failed_new.csv')
best_match_df.to_csv('~/Downloads/success_new.csv')

4204
0
4204
4204
(0, 15)
(3967, 16)


In [45]:
print(no_match.shape)
print(best_match_df.shape)
print(len(no_match['idddd'].unique()))
print(len(best_match_df['idddd'].unique()))

(237, 16)
(3967, 16)
237
3967
